# signal_data 머신러닝 분석

이 노트북은 `signal_data.csv`를 이용해 **Signaltype을 예측하는 회귀 문제**로 구성했습니다.

## 전체 흐름
1. CSV 업로드 및 불러오기
2. 데이터 구조 확인
3. 결측치·중복값 확인
4. 기초 통계 및 시각화
5. 전처리
6. 학습/테스트 데이터 분리
7. 여러 회귀 모델 비교
8. MAE, RMSE, R² 평가
9. 가장 좋은 모델 선정
10. 실제값과 예측값 비교
11. 변수 중요도 확인
12. 결론 출력

> **주의:** 현재 데이터에서 `Signaltype`은 0~97 범위의 정수형 수치로 되어 있어 회귀 문제로 가정했습니다.  
> 수업에서 `Signaltype`을 '범주형 클래스'라고 따로 정의했다면 분류 문제로 바꿔야 합니다.


## 1. 라이브러리 불러오기

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option('display.max_columns', None)


## 2. CSV 파일 업로드

In [ ]:
uploaded = files.upload()

# 업로드한 CSV 파일 이름 자동 선택
csv_name = next(name for name in uploaded.keys() if name.lower().endswith('.csv'))
df = pd.read_csv(csv_name)

print("불러온 파일:", csv_name)
print("데이터 크기:", df.shape)
df.head()


## 3. 데이터 기본 구조 확인

In [ ]:
print("행, 열 개수:", df.shape)
print("\n컬럼명:")
print(df.columns.tolist())

print("\n데이터 타입:")
print(df.dtypes)

print("\n기본 정보:")
df.info()


## 4. 결측치와 중복값 확인

In [ ]:
print("결측치 개수")
display(df.isnull().sum().to_frame("결측치 수"))

print("\n중복 행 개수:", df.duplicated().sum())


## 5. 타깃 변수 확인

여기서는 **예측하려는 값(y)** 을 `Signaltype`으로 설정합니다.

머신러닝에서 가장 중요한 것은 결국:

- 입력 변수(X)를 이용해
- 타깃 변수(y)를 얼마나 정확하게 예측하는지 확인하는 것입니다.

회귀 문제에서는 주로 **MAE, RMSE, R²**를 확인합니다.


In [ ]:
target = 'Signaltype'

print("Signaltype 최소값:", df[target].min())
print("Signaltype 최대값:", df[target].max())
print("Signaltype 고유값 개수:", df[target].nunique())
print("\nSignaltype 분포:")
display(df[target].value_counts().sort_index().head(30))


## 6. 기술통계 확인

In [ ]:
display(df.describe(include='all').T)


## 7. Signaltype 분포 시각화

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df['Signaltype'], bins=30)
plt.title('Distribution of Signaltype')
plt.xlabel('Signaltype')
plt.ylabel('Count')
plt.show()


## 8. 수치형 변수와 Signaltype의 상관관계

In [ ]:
numeric_df = df.select_dtypes(include=np.number)

corr = numeric_df.corr()

print("Signaltype과의 상관계수")
display(corr[['Signaltype']].sort_values('Signaltype', ascending=False))

plt.figure(figsize=(8, 5))
corr['Signaltype'].drop('Signaltype').sort_values().plot(kind='barh')
plt.title('Correlation with Signaltype')
plt.xlabel('Correlation')
plt.show()


## 9. 시간 변수 가공

`Time_Departure`, `Time_Arrival`은 문자열이라 그대로 모델에 넣기 어렵습니다.  
따라서 시/분/초 형태의 숫자 변수로 변환합니다.

`RID`는 각 행마다 고유한 식별자이므로 예측에 도움이 되기 어렵다고 보고 제외합니다.


In [ ]:
data = df.copy()

for col, prefix in [('Time_Departure', 'Dep'), ('Time_Arrival', 'Arr')]:
    t = pd.to_datetime(data[col], format='mixed', errors='coerce')
    data[f'{prefix}_Hour'] = t.dt.hour
    data[f'{prefix}_Minute'] = t.dt.minute
    data[f'{prefix}_Second'] = t.dt.second

data = data.drop(columns=['RID', 'Time_Departure', 'Time_Arrival'])

data.head()


## 10. 입력 변수(X)와 타깃(y) 분리

In [ ]:
X = data.drop(columns=['Signaltype'])
y = data['Signaltype']

print("X 크기:", X.shape)
print("y 크기:", y.shape)

display(X.head())


## 11. 숫자형/범주형 컬럼 구분

In [ ]:
categorical_cols = X.select_dtypes(include='object').columns.tolist()
numeric_cols = X.select_dtypes(exclude='object').columns.tolist()

print("범주형 변수:", categorical_cols)
print("수치형 변수:", numeric_cols)


## 12. 전처리 설정

- 수치형 변수: 결측치를 중앙값으로 채우고 표준화
- 범주형 변수: 최빈값으로 결측치를 채우고 One-Hot Encoding


In [ ]:
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])


## 13. 학습용/테스트용 데이터 분리

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("학습 데이터:", X_train.shape)
print("테스트 데이터:", X_test.shape)


## 14. 여러 회귀 모델 학습 및 비교

비교 모델:
- Linear Regression
- Random Forest Regressor
- Gradient Boosting Regressor


In [ ]:
models = {
    'LinearRegression': LinearRegression(),
    'RandomForest': RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    'GradientBoosting': GradientBoostingRegressor(
        random_state=42
    )
}

results = []
trained_models = {}

for name, model in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    results.append({
        'Model': name,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2
    })

    trained_models[name] = pipeline

results_df = pd.DataFrame(results).sort_values('R2', ascending=False)
display(results_df)


## 15. 평가 지표 해석

- **MAE**: 실제값과 예측값의 평균적인 차이입니다. **작을수록 좋습니다.**
- **RMSE**: 큰 오차에 더 큰 벌점을 주는 지표입니다. **작을수록 좋습니다.**
- **R²**: 모델이 데이터의 변동을 얼마나 설명하는지 나타냅니다. **1에 가까울수록 좋습니다.**

따라서 일반적으로 **R²가 가장 높고 MAE/RMSE가 낮은 모델**을 선택합니다.


## 16. 가장 좋은 모델 자동 선택

In [ ]:
best_name = results_df.iloc[0]['Model']
best_model = trained_models[best_name]

print("가장 좋은 모델:", best_name)
print(results_df.iloc[0])


## 17. 실제값과 예측값 비교

In [ ]:
best_pred = best_model.predict(X_test)

comparison = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': best_pred,
    'Error': y_test.values - best_pred
})

display(comparison.head(20))


## 18. 실제값 vs 예측값 그래프

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test, best_pred, alpha=0.4)

min_v = min(y_test.min(), best_pred.min())
max_v = max(y_test.max(), best_pred.max())
plt.plot([min_v, max_v], [min_v, max_v], linestyle='--')

plt.xlabel('Actual Signaltype')
plt.ylabel('Predicted Signaltype')
plt.title(f'Actual vs Predicted - {best_name}')
plt.show()


## 19. 오차 분포 확인

In [ ]:
residuals = y_test.values - best_pred

plt.figure(figsize=(9, 5))
plt.hist(residuals, bins=30)
plt.xlabel('Residual')
plt.ylabel('Count')
plt.title('Residual Distribution')
plt.show()


## 20. Random Forest 변수 중요도 확인

Random Forest 모델을 이용해 어떤 변수가 예측에 상대적으로 많이 사용되었는지 확인합니다.


In [ ]:
rf_pipeline = trained_models['RandomForest']

# 전처리 후 변수명 가져오기
feature_names = rf_pipeline.named_steps['preprocessor'].get_feature_names_out()

# 중요도 가져오기
importances = rf_pipeline.named_steps['model'].feature_importances_

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

display(importance_df.head(20))

plt.figure(figsize=(9, 7))
importance_df.head(15).sort_values('Importance').plot(
    x='Feature',
    y='Importance',
    kind='barh',
    legend=False
)
plt.title('Top 15 Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()


## 21. 최종 결과 자동 정리

In [ ]:
best_row = results_df.iloc[0]

print("===== 최종 머신러닝 분석 결과 =====")
print(f"예측 대상: Signaltype")
print(f"최종 선택 모델: {best_row['Model']}")
print(f"MAE : {best_row['MAE']:.4f}")
print(f"RMSE: {best_row['RMSE']:.4f}")
print(f"R²  : {best_row['R2']:.4f}")

print("\n해석:")
print("- MAE와 RMSE는 낮을수록 예측 오차가 작습니다.")
print("- R²는 1에 가까울수록 설명력이 높습니다.")
print("- 따라서 세 지표를 함께 비교하여 최종 모델의 성능을 판단합니다.")


# 제출용 결론 작성 예시

본 분석에서는 `Signaltype`을 예측 대상 변수로 설정하고 데이터 전처리 후 Linear Regression, Random Forest, Gradient Boosting 모델을 비교하였다.  
모델 평가는 MAE, RMSE, R²를 사용하였으며, R²가 높고 MAE와 RMSE가 낮은 모델을 최종 모델로 선정하였다.  
또한 실제값과 예측값 비교 및 변수 중요도 분석을 통해 모델의 예측 특성을 확인하였다.

> 실행 후 나온 실제 수치를 위 문장에 추가하면 최종 보고서 결론으로 사용할 수 있습니다.
